# 第 2 章 · 调用与流式（Invocation & Streaming）

> 对应文档：**Agents → Invocation / Streaming**
> https://docs.langchain.com/oss/python/langchain/agents#invocation
>
> 每个小节的结构都一样：**文档怎么说 → 平台上对应哪里 → 练习怎么做 → 看什么算证据**。
> Python 块是文档里的原写法，**只作对照，不用运行** —— 在这个平台上，同样的东西是
> 在 agent 编辑器的五个页签里填字段：Overview / Prompt / Tools / Skills / JSON。


## 1 · Invocation

> Invoke the agent with a dict containing `messages`. The agent runs the model/tool loop
> and returns the final state.

```python
result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather?"}]})
result["messages"][-1].content
```

**平台上对应哪里**：聊天面板。**一条消息 = 一次 invoke**，你不用自己组装 `messages`。

**这一节真正要理解的只有一件事**：invoke 返回的是**整个 state**，而不只是最后一句。
所以调用方能看到：

- 这一轮调了哪些工具、参数是什么、结果是什么；
- 中间出现过哪些消息（tool 消息、子代理活动）；
- token 用量。

**练习**：给 agent 一个工具（`python_execute`），问一道需要计算的题，然后**展开回复下面的
"Thinking / N steps"**，看它把什么当成了 state 的一部分。

**看什么算证据**：工具调用的参数和返回值都在里面。**这就是"证据"的具体位置** ——
不看这里，就只能看它自己说得好不好听。

## 2 · thread_id：对话 = thread

> A follow-up turn on the same conversation: reuse the same `thread_id` to keep history.

```python
config = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": [{"role": "user", "content": "Hi, I'm Weiping"}]}, config)
agent.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config)   # remembers
```

**平台上对应哪里**：**这次聊天本身就是 thread**。没有 `thread_id` 参数可以填，也不需要填：

| 你想要 | 平台上的做法 |
|---|---|
| 继续同一段对话（记得上下文） | 在同一个聊天里继续发 |
| 从零开始（不记得） | 点"新聊天" |

**练习**：把上面两句话在**同一个聊天**里发，然后在**新聊天**里只发第二句。

**看什么算证据**：同一聊天答得出，新聊天答不出，且新聊天第一句会带平台注入的欢迎语
（"第一句话"永远是这句话的位置——这也是为什么测"裸模型"时要把首轮先用掉）。

## 3 · Streaming

> Stream tokens (and intermediate steps) instead of waiting for the whole answer.

```python
for chunk in agent.stream({"messages": [{"role": "user", "content": "..."}]}):
    print(chunk)
```

**平台上对应哪里**：**默认就是流式**，没有开关。

**但这一节在本平台上有一个真问题值得讲：思考型模型的首字延迟。**
`deepseek-flash`（V4.1）默认开启 thinking：它先流出一段推理内容，再出正文。
在这段期间界面上是"没有反应"的 —— 看起来像卡住，其实是模型在想。

**练习**：同一个问题分别用 `deepseek-chat` 和 `deepseek-flash` 发一次，观察从发送到出现
第一个字的时间差。

**看什么算证据**：时间差不是网络问题。这类"看起来卡住"的体验，属于**模型行为**，
不是 bug；换模型或降低推理强度（reasoning effort）才解决，写 prompt 不会。

## 小结

| 文档 | 平台 | 状态 |
|---|---|---|
| `invoke` | 发一条消息 | ✅ |
| 返回整个 state（含工具调用） | 回复下的 Thinking / 工具记录 | ✅ |
| `thread_id` | 聊天会话本身 | ✅（用"新聊天"表达"新 thread"） |
| `stream` | 默认流式 | ✅ |
| 推理流与正文分开 | 思考内容先到，正文后到 | ✅（会让人以为卡住） |